In [1]:
#| default_exp mass.massmesh_module

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [3]:
#| export
import trimesh
import numpy as np
import os
import sys
from trimesh.creation import cylinder
import pandas as pd
from trimesh.visual.color import ColorVisuals
from anthropmass.mass.measurements_heights_module import *

# allow importing your measurement code
#sys.path.append(r"C:\Users\theo4\OneDrive\Skrivbord\BSPLocal copy")
#from measurements_heightsC import get_measurements


C:\Users\theo4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 2.2.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, 

AttributeError: _ARRAY_API not found

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
#| export
# === HELPERS ===
# ---------------------------------------------------------------------
# 650_mass_generate_meshes.py   —  DROP-IN REPLACEMENT FOR THE FUNCTION
# ---------------------------------------------------------------------



# ── helpers (unchanged) ───────────────────────────────────────────────
def create_truncated_cone(radius_top, radius_bottom, height, sections=32):
    angles  = np.linspace(0, 2*np.pi, sections, endpoint=False)
    top     = np.c_[radius_top*np.cos(angles),
                    radius_top*np.sin(angles),
                    np.full_like(angles,  height/2)]
    bottom  = np.c_[radius_bottom*np.cos(angles),
                    radius_bottom*np.sin(angles),
                    np.full_like(angles, -height/2)]
    verts   = np.vstack([top, bottom])
    faces   = []
    for i in range(sections):
        a, b     = i, (i+1) % sections
        c, d     = i+sections, (i+1) % sections + sections
        faces   += [[a, b, c], [b, d, c]]
    # cap
    verts  = np.vstack([verts, [[0,0, height/2],[0,0,-height/2]]])
    top_c  = len(verts)-2; bot_c = len(verts)-1
    for i in range(sections):
        nxt = (i+1) % sections
        faces += [[i, nxt, top_c], [nxt+sections, i+sections, bot_c]]
    return trimesh.Trimesh(vertices=verts, faces=faces, process=True)

def create_elliptical_frustum_solid(a_top,b_top,a_bot,b_bot,h,n=32):
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    top    = np.c_[a_top*np.cos(angles), b_top*np.sin(angles), np.full_like(angles,  h/2)]
    bot    = np.c_[a_bot*np.cos(angles), b_bot*np.sin(angles), np.full_like(angles, -h/2)]
    verts  = np.vstack([top, bot])
    faces  = []
    for i in range(n):
        a,b  = i,(i+1)%n
        c,d  = i+n,(i+1)%n+n
        faces += [[a,b,c],[b,d,c]]
    verts  = np.vstack([verts, [[0,0,h/2],[0,0,-h/2]]])
    top_c  = len(verts)-2; bot_c=len(verts)-1
    for i in range(n):
        nxt  = (i+1)%n
        faces += [[i,nxt,top_c],[nxt+n,i+n,bot_c]]
    return trimesh.Trimesh(vertices=verts, faces=faces, process=True)

def create_elliptical_cylinder(a,b,h,sections=32):
    return create_elliptical_frustum_solid(a,b,a,b,h,sections)

def mirror_mesh(mesh):
    m = mesh.copy(); m.apply_scale([-1,1,1]); return m

def rotate_hand_mesh(mesh):
    rot_z = trimesh.transformations.rotation_matrix(np.pi/2,[0,0,1])
    mesh.apply_transform(rot_z); return mesh

def rotate_foot_mesh(mesh):
    rot_x = trimesh.transformations.rotation_matrix(np.pi/2,[-1,0,0])
    rot_z = trimesh.transformations.rotation_matrix(np.pi/2,[0,0,1])
    mesh.apply_transform(rot_x); mesh.apply_transform(rot_z); return mesh

def center_mesh(mesh):
    zmin,zmax = mesh.bounds[:,2]
    mesh.apply_translation([0,0,-(zmin+zmax)/2]); return mesh
# ──────────────────────────────────────────────────────────────────────


def generate_all_meshes(Ansur, inputheight):
    """
    Build every body-segment mesh **centred at the origin**.
    The URDF will later position them, so we do *no* height
    translations here.
    """
    # ── 1. measurements ------------------------------------------------
    if "a1" in Ansur.columns:            # already processed row
        meas = Ansur.iloc[0]
    else:                                # raw ANSUR record
        meas = get_measurements(Ansur, inputheight).iloc[0]

    out_dir = os.path.join(os.path.dirname(__file__), "model_output")
    os.makedirs(out_dir, exist_ok=True)

    def save(mesh,name):
        mesh.fix_normals()
        mesh.export(os.path.join(out_dir,f"{name}.stl"))

    # ── 2. torso + neck -----------------------------------------------
    torso = {
        "torso_top":    create_elliptical_cylinder(meas["a1"], meas["b1"], meas["h1"]),
        "torso_mid":    create_elliptical_frustum_solid(meas["a2"],meas["b2"],
                                                        meas["a3"],meas["b3"],meas["h2"]),
        "torso_bottom": create_elliptical_cylinder(meas["a4"], meas["b4"], meas["h3"]),
        "neck":         cylinder(radius=meas["a9"], height=meas["a8"], sections=32),
    }
    for name,mesh in torso.items():
        save(center_mesh(mesh), name)

    # ── 3. head --------------------------------------------------------
    head = trimesh.primitives.Sphere(radius=1).to_mesh()
    head.apply_transform(np.diag([meas["b8"],meas["b8"],meas["a7"],1]))
    save(center_mesh(head), "head")

    # ── 4. limbs -------------------------------------------------------
    limbs = {
        "upper_arm": ("trunc", (meas["r1"],        meas["r2"],        meas["h8"])),
        "forearm":   ("trunc", (meas["r3"],        meas["r4"],        meas["h10"])),
        "upper_leg": ("trunc", (meas["r1thigh"],   meas["r2thigh"],   meas["h4"])),
        "lower_leg": ("trunc", (meas["r1shank"],   meas["r2shank"],   meas["h6"])),
        "hand":      ("ellip", [meas["r7"],        meas["b7"],        meas["h9"]]),
        "foot":      ("frust",( meas["b5"],        meas["a5"],
                                meas["a6"],        meas["b6"],        meas["h7"])),
    }

    for part,(kind,dims) in limbs.items():
        for side in ("left","right"):
            # build base geometry
            if   kind=="trunc":
                rt,rb,h = dims; m = create_truncated_cone(rt,rb,h)
            elif kind=="ellip":
                m = trimesh.primitives.Sphere(radius=1).to_mesh()
                m.apply_transform(np.diag(dims+[1]))
            else:                       # foot
                at,bt,ab,bb,h = dims
                m = create_elliptical_frustum_solid(at,bt,ab,bb,h)

            # rotations
            if   part=="hand": m = rotate_hand_mesh(m)
            elif part=="foot": m = rotate_foot_mesh(m)

            m = center_mesh(m)          # final centre
            if side=="right": m = mirror_mesh(m)
            save(m, f"{part}_{side}")

    print(f"✅ Meshes exported to {out_dir}")










In [ ]:
#| export


# === FUNCTION: Preview Scene ===
def preview_current_colored_geometry(Ansur, inputheight):
    measurements = get_measurements(Ansur, inputheight).iloc[0]

    torso_top_a = measurements["a1"]
    torso_top_b = measurements["b1"]
    torso_capsule_height_top = measurements["h1"]
    torso_mid_top_a = measurements["a2"]
    torso_mid_top_b = measurements["b2"]
    torso_mid_bot_a = measurements["a3"]
    torso_mid_bot_b = measurements["b3"]
    torso_mid_height = measurements["h2"]
    torso_bot_a = measurements["a4"]
    torso_bot_b = measurements["b4"]
    torso_capsule_height_bottom = measurements["h3"]
    neck_radius = measurements["a9"]
    neck_height = measurements["a8"]/5
    
    head_radii = [measurements["b8"], measurements["b8"], measurements["a7"]]
    upper_arm_radius_top = measurements["r1"]
    upper_arm_radius_bottom = measurements["r2"]
    upper_arm_length = measurements["h8"]
    forearm_radius_top = measurements["r3"]
    forearm_radius_bottom = measurements["r4"]
    forearm_length = measurements["h10"]
    upper_leg_radius_top = measurements["r1thigh"]
    upper_leg_radius_bottom = measurements["r2thigh"]
    upper_leg_length = measurements["h4"]
    lower_leg_radius_top = measurements["r1shank"]
    lower_leg_radius_bottom = measurements["r2shank"]
    lower_leg_length = measurements["h6"]
    foot_top_b = measurements["a5"]
    foot_top_a = measurements["b5"]
    foot_bot_a = measurements["a6"]
    foot_bot_b = measurements["b6"]
    foot_height = measurements["h7"]
    hand_radii = [measurements["r7"], measurements["b7"], measurements["h9"]]

    scene = trimesh.Scene()
    x_offset = 0

    torso_parts = {
        "torso_top": (create_elliptical_cylinder(torso_top_a, torso_top_b, torso_capsule_height_top), [200, 200, 200, 255]),
        "torso_mid": (create_elliptical_frustum_solid(torso_mid_top_a, torso_mid_top_b, torso_mid_bot_a, torso_mid_bot_b, torso_mid_height), [150, 150, 150, 255]),
        "torso_bottom": (create_elliptical_cylinder(torso_bot_a, torso_bot_b, torso_capsule_height_bottom), [200, 200, 200, 255]),
        "neck": (cylinder(radius=neck_radius, height=neck_height, sections=32), [255, 255, 100, 255]),
    }

    for name, (mesh, color) in torso_parts.items():
        mesh.visual = ColorVisuals(mesh, face_colors=color)
   
        mesh.apply_translation([x_offset, 0, 0])
        scene.add_geometry(mesh, node_name=name)
        x_offset += 0.6

    limb_parts = {
        "head": ("ellipsoid", head_radii, [255, 255, 255, 255]),
        "upper_arm_left": ("trunc_cone", (upper_arm_radius_top, upper_arm_radius_bottom, upper_arm_length), [255, 0, 0, 255]),
        "forearm_left": ("trunc_cone", (forearm_radius_top, forearm_radius_bottom, forearm_length), [255, 165, 0, 255]),
        "upper_leg_left": ("trunc_cone", (upper_leg_radius_top, upper_leg_radius_bottom, upper_leg_length), [0, 0, 255, 255]),
        "lower_leg_left": ("trunc_cone", (lower_leg_radius_top, lower_leg_radius_bottom, lower_leg_length), [0, 255, 0, 255]),
        "foot_left": ("elliptical_frustum", (foot_top_a, foot_top_b, foot_bot_a, foot_bot_b, foot_height), [160, 82, 45, 255]),
        "hand_left": ("ellipsoid", hand_radii, [255, 192, 203, 255]),
    }

    for part, (shape_type, dims, color) in limb_parts.items():
        if shape_type == "ellipsoid":
            mesh = trimesh.primitives.Sphere(radius=1.0).to_mesh()
            mesh.apply_transform(np.diag(dims + [1.0]))
        elif shape_type == "trunc_cone":
            rt, rb, h = dims
            mesh = create_truncated_cone(rt, rb, h)
            mesh.apply_translation([0, 0, -h / 2])
        elif shape_type == "elliptical_frustum":
            at, bt, ab, bb, h = dims
            mesh = create_elliptical_frustum_solid(at, bt, ab, bb, h)
        else:
            continue
        mesh.visual = ColorVisuals(mesh, face_colors=color)
        mesh.apply_translation([x_offset, 0, 0])
        scene.add_geometry(mesh, node_name=part)
        x_offset += 0.6
        

        # ... your existing mesh creation
        if part == "foot_left":
            angle = np.pi / 2  # 90 degrees
            rotation = trimesh.transformations.rotation_matrix(
                angle=angle,
                direction=[-1, 0, 0],  
                point=[0, 0, 0]
            )
            mesh.apply_transform(rotation)

             # ... your existing mesh creation
        if part == "foot_left":
            angle = np.pi / 2  # 90 degrees
            rotation = trimesh.transformations.rotation_matrix(
                angle=angle,
                direction=[0, 0, 1],  # Z-axis
                point=[0, 0, 0]
            )
            mesh.apply_transform(rotation)


    scene.show(flags={'cull': False})

# === MAIN ===
Ansur = pd.read_csv('../data/processed/ANSURIImalefemale.csv')
subject = Ansur.iloc[[0]]  # Use one subject
generate_all_meshes(subject, 1800)
#preview_current_colored_geometry(subject, 1800)





# 🟥 Red = Upper Arm
# 🟧 Orange = Forearm
# 🟦 Blue = Upper Leg
# 🟩 Green = Lower Leg
# ⚪ White = Head
# ⬜ Gray = Torso

In [4]:
import nbdev; nbdev.nbdev_export()